# Taal Vista Hotel Data Cleaning

## Purpose

This notebook cleans and validates the raw datasets collected from the official Taal Vista Hotel website and its official booking portal.

The cleaning process will:

1. Load and inspect every raw CSV file.
2. Standardize column names and text formatting.
3. Assign appropriate data types.
4. Identify valid and problematic missing values.
5. Check duplicate records.
6. Resolve formatting inconsistencies without changing source facts.
7. Preserve documented conflicts between sources.
8. Save cleaned datasets in the `data/clean` folder.

Review testimonials displayed on the official hotel website will be treated as curated excerpts. They will not be interpreted as a representative sample of all guest reviews.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

In [2]:
project_root = Path.cwd().parent

raw_data_folder = (
    project_root
    / "data"
    / "raw"
)

clean_data_folder = (
    project_root
    / "data"
    / "clean"
)

clean_data_folder.mkdir(
    parents=True,
    exist_ok=True
)

print("Project folder:", project_root)
print("Raw data folder:", raw_data_folder)
print("Clean data folder:", clean_data_folder)
print("Raw folder exists:", raw_data_folder.exists())
print("Clean folder exists:", clean_data_folder.exists())

Project folder: /Users/jannoelvero/Documents/Taal-Vista-Hotel
Raw data folder: /Users/jannoelvero/Documents/Taal-Vista-Hotel/data/raw
Clean data folder: /Users/jannoelvero/Documents/Taal-Vista-Hotel/data/clean
Raw folder exists: True
Clean folder exists: True


In [3]:
raw_csv_files = sorted(
    raw_data_folder.glob("*.csv")
)

raw_inventory_records = []

for csv_file in raw_csv_files:
    current_df = pd.read_csv(
        csv_file
    )

    raw_inventory_records.append({
        "file_name": csv_file.name,
        "rows": current_df.shape[0],
        "columns": current_df.shape[1],
        "missing_cells": (
            current_df.isna().sum().sum()
        ),
        "duplicate_rows": (
            current_df.duplicated().sum()
        )
    })

raw_inventory_df = pd.DataFrame(
    raw_inventory_records
)

print(
    "Raw CSV files:",
    len(raw_csv_files)
)

print(
    "Total stored rows:",
    raw_inventory_df["rows"].sum()
)

print(
    "Files with duplicate rows:",
    (
        raw_inventory_df["duplicate_rows"] > 0
    ).sum()
)

display(raw_inventory_df)

Raw CSV files: 21
Total stored rows: 397
Files with duplicate rows: 0


,file_name,rows,columns,missing_cells,duplicate_rows
0,taal_vista_christmas_packages_raw.csv,4,19,0,0
1,taal_vista_dining_raw.csv,5,10,9,0
2,taal_vista_events_lake_desktop_raw.csv,17,14,14,0
3,taal_vista_events_lake_mobile_raw.csv,14,14,11,0
4,taal_vista_events_mountain_desktop_raw.csv,11,14,13,0
5,taal_vista_events_mountain_mobile_raw.csv,9,14,13,0
6,taal_vista_facilities_raw.csv,5,6,0,0
7,taal_vista_hotel_contact_raw.csv,5,8,4,0
8,taal_vista_hotel_facts_raw.csv,12,8,5,0
9,taal_vista_meeting_rates_raw.csv,18,16,6,0


In [4]:
supporting_room_rate_files = [
    "taal_vista_room_rates_search_1_raw.csv",
    "taal_vista_room_rates_search_2_raw.csv",
    "taal_vista_room_rates_search_3_raw.csv",
    "taal_vista_room_rates_search_4_raw.csv"
]

raw_inventory_df["file_role"] = (
    raw_inventory_df["file_name"].apply(
        lambda file_name: (
            "Supporting extraction file"
            if file_name
            in supporting_room_rate_files
            else "Primary dataset"
        )
    )
)

display(
    raw_inventory_df[
        [
            "file_name",
            "file_role",
            "rows",
            "columns",
            "missing_cells",
            "duplicate_rows"
        ]
    ]
)

,file_name,file_role,rows,columns,missing_cells,duplicate_rows
0,taal_vista_christmas_packages_raw.csv,Primary dataset,4,19,0,0
1,taal_vista_dining_raw.csv,Primary dataset,5,10,9,0
2,taal_vista_events_lake_desktop_raw.csv,Primary dataset,17,14,14,0
3,taal_vista_events_lake_mobile_raw.csv,Primary dataset,14,14,11,0
4,taal_vista_events_mountain_desktop_raw.csv,Primary dataset,11,14,13,0
5,taal_vista_events_mountain_mobile_raw.csv,Primary dataset,9,14,13,0
6,taal_vista_facilities_raw.csv,Primary dataset,5,6,0,0
7,taal_vista_hotel_contact_raw.csv,Primary dataset,5,8,4,0
8,taal_vista_hotel_facts_raw.csv,Primary dataset,12,8,5,0
9,taal_vista_meeting_rates_raw.csv,Primary dataset,18,16,6,0


In [5]:
primary_inventory_df = (
    raw_inventory_df[
        raw_inventory_df["file_role"]
        == "Primary dataset"
    ]
    .copy()
)

print(
    "Primary datasets:",
    primary_inventory_df.shape[0]
)

print(
    "Primary analytical rows:",
    primary_inventory_df["rows"].sum()
)

print(
    "Supporting extraction files:",
    (
        raw_inventory_df["file_role"]
        == "Supporting extraction file"
    ).sum()
)

print(
    "Supporting extraction rows:",
    raw_inventory_df.loc[
        raw_inventory_df["file_role"]
        == "Supporting extraction file",
        "rows"
    ].sum()
)

Primary datasets: 17
Primary analytical rows: 304
Supporting extraction files: 4
Supporting extraction rows: 93


In [6]:
primary_dataframes = {}

for file_name in primary_inventory_df[
    "file_name"
]:
    dataset_name = (
        file_name
        .replace("taal_vista_", "")
        .replace("_raw.csv", "")
    )

    file_path = (
        raw_data_folder
        / file_name
    )

    primary_dataframes[dataset_name] = (
        pd.read_csv(file_path)
    )

print(
    "Primary datasets loaded:",
    len(primary_dataframes)
)

for dataset_name, current_df in (
    primary_dataframes.items()
):
    print(
        dataset_name,
        current_df.shape
    )

Primary datasets loaded: 17
christmas_packages (4, 19)
dining (5, 10)
events_lake_desktop (17, 14)
events_lake_mobile (14, 14)
events_mountain_desktop (11, 14)
events_mountain_mobile (9, 14)
facilities (5, 6)
hotel_contact (5, 8)
hotel_facts (12, 8)
meeting_rates (18, 16)
official_reviews (6, 10)
pasalubong_prices (34, 13)
promotions (25, 9)
room_rates (93, 33)
rooms (11, 5)
spa_prices (26, 12)
wedding_rates (9, 13)


In [7]:
schema_records = []

for dataset_name, current_df in (
    primary_dataframes.items()
):
    for column_name in current_df.columns:
        schema_records.append({
            "dataset_name": dataset_name,
            "column_name": column_name,
            "data_type": str(
                current_df[column_name].dtype
            ),
            "missing_values": (
                current_df[column_name]
                .isna()
                .sum()
            ),
            "unique_values": (
                current_df[column_name]
                .nunique(
                    dropna=True
                )
            )
        })

raw_schema_df = pd.DataFrame(
    schema_records
)

print(
    "Schema records:",
    raw_schema_df.shape[0]
)

display(raw_schema_df)

Schema records: 218


,dataset_name,column_name,data_type,missing_values,unique_values
0,christmas_packages,christmas_package_record_id,int64,0,4
1,christmas_packages,campaign_name,str,0,1
2,christmas_packages,package_name,str,0,4
3,christmas_packages,package_price_php,int64,0,4
4,christmas_packages,guaranteed_persons,int64,0,4
...,...,...,...,...,...
213,wedding_rates,currency,str,0,1
214,wedding_rates,price_basis,str,0,2
215,wedding_rates,verification_method,str,0,1
216,wedding_rates,source_url,str,0,1


In [8]:
import re

clean_dataframes = {}

for dataset_name, current_df in (
    primary_dataframes.items()
):
    clean_dataframes[dataset_name] = (
        current_df.copy()
    )

print(
    "Cleaning copies created:",
    len(clean_dataframes)
)

Cleaning copies created: 17


In [9]:
for dataset_name, current_df in (
    clean_dataframes.items()
):
    current_df.columns = (
        current_df.columns
        .str.strip()
        .str.lower()
        .str.replace(
            " ",
            "_"
        )
    )

    for column_name in current_df.columns:
        if (
            current_df[column_name].dtype
            == "object"
            or str(
                current_df[column_name].dtype
            ) == "str"
        ):
            current_df[column_name] = (
                current_df[column_name]
                .apply(
                    lambda value: (
                        re.sub(
                            r"\s+",
                            " ",
                            value.replace(
                                "\xa0",
                                " "
                            )
                        ).strip()
                        if isinstance(value, str)
                        else value
                    )
                )
                .replace(
                    "",
                    pd.NA
                )
            )

print(
    "Text standardization complete"
)

Text standardization complete


In [10]:
column_validation_records = []

for dataset_name, current_df in (
    clean_dataframes.items()
):
    column_validation_records.append({
        "dataset_name": dataset_name,
        "columns": current_df.shape[1],
        "duplicate_column_names": (
            current_df.columns.duplicated().sum()
        )
    })

column_validation_df = pd.DataFrame(
    column_validation_records
)

display(column_validation_df)

print(
    "Datasets with duplicate column names:",
    (
        column_validation_df[
            "duplicate_column_names"
        ] > 0
    ).sum()
)

,dataset_name,columns,duplicate_column_names
0,christmas_packages,19,0
1,dining,10,0
2,events_lake_desktop,14,0
3,events_lake_mobile,14,0
4,events_mountain_desktop,14,0
5,events_mountain_mobile,14,0
6,facilities,6,0
7,hotel_contact,8,0
8,hotel_facts,8,0
9,meeting_rates,16,0


Datasets with duplicate column names: 0


In [11]:
date_conversion_records = []

for dataset_name, current_df in (
    clean_dataframes.items()
):
    date_columns = [
        column_name
        for column_name in current_df.columns
        if (
            column_name.endswith("_date")
            or column_name
            == "date_collected"
            or column_name
            == "early_bird_deadline"
            or column_name
            == "valid_until"
        )
    ]

    for column_name in date_columns:
        original_missing = (
            current_df[column_name]
            .isna()
            .sum()
        )

        current_df[column_name] = (
            pd.to_datetime(
                current_df[column_name],
                errors="coerce"
            )
        )

        final_missing = (
            current_df[column_name]
            .isna()
            .sum()
        )

        date_conversion_records.append({
            "dataset_name": dataset_name,
            "column_name": column_name,
            "data_type": str(
                current_df[column_name].dtype
            ),
            "original_missing": original_missing,
            "final_missing": final_missing,
            "invalid_dates_created": (
                final_missing
                - original_missing
            )
        })

date_conversion_df = pd.DataFrame(
    date_conversion_records
)

display(date_conversion_df)

print(
    "Date columns converted:",
    date_conversion_df.shape[0]
)

print(
    "Invalid dates created:",
    date_conversion_df[
        "invalid_dates_created"
    ].sum()
)

,dataset_name,column_name,data_type,original_missing,final_missing,invalid_dates_created
0,christmas_packages,early_bird_deadline,datetime64[us],0,0,0
1,christmas_packages,valid_until,datetime64[us],0,0,0
2,christmas_packages,date_collected,datetime64[us],0,0,0
3,dining,date_collected,datetime64[us],0,0,0
4,events_lake_desktop,date_collected,datetime64[us],0,0,0
5,events_lake_mobile,date_collected,datetime64[us],0,0,0
6,events_mountain_desktop,date_collected,datetime64[us],0,0,0
7,events_mountain_mobile,date_collected,datetime64[us],0,0,0
8,facilities,date_collected,datetime64[us],0,0,0
9,hotel_contact,date_collected,datetime64[us],0,0,0


Date columns converted: 22
Invalid dates created: 0


In [12]:
boolean_audit_records = []

boolean_keywords = [
    "inclusive",
    "included",
    "allowed",
    "required",
    "truncated"
]

for dataset_name, current_df in (
    clean_dataframes.items()
):
    for column_name in current_df.columns:
        if any(
            keyword in column_name
            for keyword in boolean_keywords
        ):
            unique_values = (
                current_df[column_name]
                .dropna()
                .astype(str)
                .unique()
                .tolist()
            )

            boolean_audit_records.append({
                "dataset_name": dataset_name,
                "column_name": column_name,
                "current_type": str(
                    current_df[column_name].dtype
                ),
                "missing_values": (
                    current_df[column_name]
                    .isna()
                    .sum()
                ),
                "unique_values": unique_values
            })

boolean_audit_df = pd.DataFrame(
    boolean_audit_records
)

display(boolean_audit_df)

,dataset_name,column_name,current_type,missing_values,unique_values
0,christmas_packages,tax_inclusive,bool,0,[True]
1,official_reviews,excerpt_truncated,bool,0,"[False, True]"
2,pasalubong_prices,tax_inclusive,bool,0,[True]
3,room_rates,breakfast_included,bool,0,[True]
4,room_rates,modification_allowed,bool,0,"[True, False]"
5,spa_prices,tax_inclusive,bool,0,[True]
6,wedding_rates,included_persons,float64,1,[100.0]


In [13]:
print(
    clean_dataframes[
        "room_rates"
    ].columns.tolist()
)

['room_rate_record_id', 'room_name', 'room_size_sqm', 'starting_price_php', 'availability_note', 'search_id', 'check_in_date', 'check_out_date', 'day_type', 'number_of_nights', 'number_of_rooms', 'number_of_adults', 'number_of_children', 'currency', 'rate_display_type', 'tax_status', 'booking_url', 'collection_timestamp', 'verification_method', 'rate_plan_name', 'rate_plan_price_php', 'breakfast_included', 'cancellation_policy', 'room_size_note', 'cancellation_window_days', 'modification_allowed', 'payment_requirement', 'late_cancellation_penalty', 'check_in_time', 'check_out_time', 'children_policy', 'rate_inclusions', 'rate_details_text']


In [14]:
room_rates_clean_df = (
    clean_dataframes["room_rates"]
)

original_missing = (
    room_rates_clean_df[
        "collection_timestamp"
    ].isna().sum()
)

room_rates_clean_df[
    "collection_timestamp"
] = pd.to_datetime(
    room_rates_clean_df[
        "collection_timestamp"
    ],
    errors="coerce"
)

final_missing = (
    room_rates_clean_df[
        "collection_timestamp"
    ].isna().sum()
)

print(
    "Data type:",
    room_rates_clean_df[
        "collection_timestamp"
    ].dtype
)

print(
    "Original missing:",
    original_missing
)

print(
    "Final missing:",
    final_missing
)

print(
    "Invalid timestamps created:",
    final_missing - original_missing
)

Data type: datetime64[us, UTC+02:00]
Original missing: 0
Final missing: 77
Invalid timestamps created: 77


In [15]:
numeric_audit_records = []

for dataset_name, current_df in (
    clean_dataframes.items()
):
    numeric_columns = (
        current_df.select_dtypes(
            include="number"
        ).columns
    )

    for column_name in numeric_columns:
        numeric_audit_records.append({
            "dataset_name": dataset_name,
            "column_name": column_name,
            "data_type": str(
                current_df[column_name].dtype
            ),
            "missing_values": (
                current_df[column_name]
                .isna()
                .sum()
            ),
            "minimum_value": (
                current_df[column_name]
                .min()
            ),
            "maximum_value": (
                current_df[column_name]
                .max()
            )
        })

numeric_audit_df = pd.DataFrame(
    numeric_audit_records
)

print(
    "Numeric columns found:",
    numeric_audit_df.shape[0]
)

display(numeric_audit_df)

Numeric columns found: 60


,dataset_name,column_name,data_type,missing_values,minimum_value,maximum_value
0,christmas_packages,christmas_package_record_id,int64,0,1.00,4.0
1,christmas_packages,package_price_php,int64,0,39000.00,225000.0
2,christmas_packages,guaranteed_persons,int64,0,20.00,100.0
3,christmas_packages,excess_person_rate_php,int64,0,1800.00,2350.0
4,christmas_packages,source_page,int64,0,1.00,2.0
5,christmas_packages,venue_usage_hours,int64,0,5.00,5.0
6,christmas_packages,kids_rate_percent,int64,0,50.00,50.0
7,christmas_packages,kultura_discount_percent,int64,0,10.00,10.0
8,dining,outlet_record_id,int64,0,1.00,5.0
9,events_lake_mobile,floor_area_(in_sq._meters),float64,2,3.25,5.5


### Currency Conversion Method

The original published prices are retained in Philippine pesos. Approximate euro and US dollar values are added to support international price comparison.

The conversion uses the European Central Bank reference rates published on September 2, 2026.

1 EUR equals 72.415 PHP  
1 EUR equals 1.1578 USD

Converted amounts are analytical estimates and may differ from actual card, bank, or booking portal conversion rates.

In [16]:
fx_rate_date = pd.Timestamp(
    "2026-09-02"
)

php_per_eur = 72.415
usd_per_eur = 1.1578

php_to_eur = 1 / php_per_eur
php_to_usd = (
    usd_per_eur
    / php_per_eur
)

print(
    "1 PHP in EUR:",
    round(php_to_eur, 6)
)

print(
    "1 PHP in USD:",
    round(php_to_usd, 6)
)

1 PHP in EUR: 0.013809
1 PHP in USD: 0.015988


In [17]:
price_columns_by_dataset = {
    "christmas_packages": [
        "package_price_php",
        "excess_person_rate_php"
    ],
    "meeting_rates": [
        "price_php"
    ],
    "pasalubong_prices": [
        "price_php"
    ],
    "room_rates": [
        "starting_price_php",
        "rate_plan_price_php"
    ],
    "spa_prices": [
        "price_php"
    ],
    "wedding_rates": [
        "package_price_php",
        "excess_person_rate_php"
    ]
}

for dataset_name, price_columns in (
    price_columns_by_dataset.items()
):
    current_df = clean_dataframes[
        dataset_name
    ]

    for php_column in price_columns:
        eur_column = php_column.replace(
            "_php",
            "_eur"
        )

        usd_column = php_column.replace(
            "_php",
            "_usd"
        )

        current_df[eur_column] = (
            current_df[php_column]
            * php_to_eur
        ).round(2)

        current_df[usd_column] = (
            current_df[php_column]
            * php_to_usd
        ).round(2)

    current_df["fx_rate_date"] = (
        fx_rate_date
    )

    current_df["fx_rate_source"] = (
        "European Central Bank"
    )

print(
    "Currency conversion complete"
)

Currency conversion complete


In [18]:
print(
    clean_dataframes["room_rates"][
        [
            "room_name",
            "rate_plan_price_php",
            "rate_plan_price_usd",
            "rate_plan_price_eur",
            "fx_rate_date"
        ]
    ].head()
)

       room_name  rate_plan_price_php  rate_plan_price_usd  \
0  Superior King                11900               190.26   
1  Superior Twin                11900               190.26   
2    Deluxe King                12900               206.25   
3   Deluxe Queen                12900               206.25   
4    Deluxe Twin                12900               206.25   

   rate_plan_price_eur fx_rate_date  
0               164.33   2026-09-02  
1               164.33   2026-09-02  
2               178.14   2026-09-02  
3               178.14   2026-09-02  
4               178.14   2026-09-02  


In [19]:
missing_value_records = []

for dataset_name, current_df in (
    clean_dataframes.items()
):
    for column_name in current_df.columns:
        missing_count = (
            current_df[column_name]
            .isna()
            .sum()
        )

        if missing_count > 0:
            missing_value_records.append({
                "dataset_name": dataset_name,
                "column_name": column_name,
                "missing_values": missing_count,
                "total_rows": current_df.shape[0],
                "missing_percent": round(
                    missing_count
                    / current_df.shape[0]
                    * 100,
                    2
                )
            })

missing_value_audit_df = pd.DataFrame(
    missing_value_records
)

print(
    "Columns with missing values:",
    missing_value_audit_df.shape[0]
)

print(
    "Total missing cells:",
    missing_value_audit_df[
        "missing_values"
    ].sum()
)

display(
    missing_value_audit_df.sort_values(
        [
            "dataset_name",
            "column_name"
        ]
    )
)

Columns with missing values: 40
Total missing cells: 434


,dataset_name,column_name,missing_values,total_rows,missing_percent
2,dining,reservation_url,3,5,60.00
1,dining,schedule_note,4,5,80.00
0,dining,tagline,2,5,40.00
3,events_lake_desktop,2,3,17,17.65
4,events_lake_desktop,5,4,17,23.53
5,events_lake_desktop,6,4,17,23.53
6,events_lake_desktop,7,3,17,17.65
10,events_lake_mobile,block,3,14,21.43
8,events_lake_mobile,classroom,3,14,21.43
7,events_lake_mobile,floor_area_(in_sq._meters),2,14,14.29


In [20]:
event_dataset_names = [
    "events_lake_desktop",
    "events_lake_mobile",
    "events_mountain_desktop",
    "events_mountain_mobile"
]

for dataset_name in event_dataset_names:
    current_df = clean_dataframes[
        dataset_name
    ]

    print()
    print("=" * 70)
    print("DATASET:", dataset_name)

    print(
        "Columns:",
        current_df.columns.tolist()
    )

    display(
        current_df.head(5)
    )


DATASET: events_lake_desktop
Columns: ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', 'hotel_wing', 'display_version', 'source_url', 'date_collected']


,0,1,2,3,4,5,6,7,8,9,hotel_wing,display_version,source_url,date_collected
0,VENUE,DIMENSION (IN METERS),CEILING HEIGHT (IN METERS),FLOOR AREA (IN SQ. METERS),CONFERENCE CAPACITY,CONFERENCE CAPACITY,CONFERENCE CAPACITY,CONFERENCE CAPACITY,BANQUET,COCKTAIL,Lake Wing,Desktop,https://www.taalvistahotel.com/events/,2026-09-02
1,VENUE,DIMENSION (IN METERS),CEILING HEIGHT (IN METERS),FLOOR AREA (IN SQ. METERS),THEATER,CLASSROOM,U-SHAPE,BLOCK,BANQUET,COCKTAIL,Lake Wing,Desktop,https://www.taalvistahotel.com/events/,2026-09-02
2,GRAND BALLROOM,43.80 X 29.50,5.50,1292.10,1000,750,200,260,800,1400,Lake Wing,Desktop,https://www.taalvistahotel.com/events/,2026-09-02
3,BALLROOM 1,29.50 X 14.60,5.50,430.70,350,220,80,100,220,450,Lake Wing,Desktop,https://www.taalvistahotel.com/events/,2026-09-02
4,BALLROOM 2,29.50 X 14.60,5.50,430.70,350,220,80,100,220,450,Lake Wing,Desktop,https://www.taalvistahotel.com/events/,2026-09-02



DATASET: events_lake_mobile
Columns: ['venue', 'dimension_(in_meters)', 'floor_area_(in_sq._meters)', 'conference_capacity', 'theater', 'classroom', 'u-shape', 'block', 'banquet', 'cocktail', 'hotel_wing', 'display_version', 'source_url', 'date_collected']


,venue,dimension_(in_meters),floor_area_(in_sq._meters),conference_capacity,theater,classroom,u-shape,block,banquet,cocktail,hotel_wing,display_version,source_url,date_collected
0,GRAND BALLROOM,43.80 X 29.50,5.5,1292.10,360,180.0,90.0,90.0,270,270,Lake Wing,Mobile,https://www.taalvistahotel.com/events/,2026-09-02
1,BALLROOM 1,29.50 X 14.60,5.5,430.70,120,60.0,30.0,30.0,90,90,Lake Wing,Mobile,https://www.taalvistahotel.com/events/,2026-09-02
2,BALLROOM 2,29.50 X 14.60,5.5,430.70,120,60.0,30.0,30.0,90,90,Lake Wing,Mobile,https://www.taalvistahotel.com/events/,2026-09-02
3,BALLROOM 3,29.50 X 14.60,5.5,430.70,120,60.0,30.0,30.0,90,90,Lake Wing,Mobile,https://www.taalvistahotel.com/events/,2026-09-02
4,COVERED TERRACE,37.90 X 9.60,3.9,363.84,100,NaN,NaN,NaN,80,80,Lake Wing,Mobile,https://www.taalvistahotel.com/events/,2026-09-02



DATASET: events_mountain_desktop
Columns: ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', 'hotel_wing', 'display_version', 'source_url', 'date_collected']


,0,1,2,3,4,5,6,7,8,9,hotel_wing,display_version,source_url,date_collected
0,VENUE,DIMENSION (IN METERS),CEILING HEIGHT (IN METERS),FLOOR AREA (IN SQ. METERS),CONFERENCE CAPACITY,CONFERENCE CAPACITY,CONFERENCE CAPACITY,CONFERENCE CAPACITY,BANQUET,COCKTAIL,Mountain Wing,Desktop,https://www.taalvistahotel.com/events/,2026-09-02
1,VENUE,DIMENSION (IN METERS),CEILING HEIGHT (IN METERS),FLOOR AREA (IN SQ. METERS),THEATER,CLASSROOM,U-SHAPE,BLOCK,BANQUET,COCKTAIL,Mountain Wing,Desktop,https://www.taalvistahotel.com/events/,2026-09-02
2,SAMPAGUITA BALLROOM,30.77 X 18.70,3.90,575.40,420,320,100,120,250,550,Mountain Wing,Desktop,https://www.taalvistahotel.com/events/,2026-09-02
3,SAMPAGUITA FOYER,15.30 X 4.60,3.88,70.38,NaN,NaN,NaN,NaN,NaN,70,Mountain Wing,Desktop,https://www.taalvistahotel.com/events/,2026-09-02
4,LOWER LEVEL FOYER,22.20 X 5.45,3.40,120.9960,100,NaN,NaN,NaN,60,120,Mountain Wing,Desktop,https://www.taalvistahotel.com/events/,2026-09-02



DATASET: events_mountain_mobile
Columns: ['venue', 'dimension_(in_meters)', 'floor_area_(in_sq._meters)', 'conference_capacity', 'theater', 'classroom', 'u-shape', 'block', 'banquet', 'cocktail', 'hotel_wing', 'display_version', 'source_url', 'date_collected']


,venue,dimension_(in_meters),floor_area_(in_sq._meters),conference_capacity,theater,classroom,u-shape,block,banquet,cocktail,hotel_wing,display_version,source_url,date_collected
0,SAMPAGUITA BALLROOM,30.77 X 18.70,3.90,575.400,120.0,80.0,30.0,30.0,100.0,100.0,Mountain Wing,Mobile,https://www.taalvistahotel.com/events/,2026-09-02
1,SAMPAGUITA FOYER,15.30 X 4.60,3.88,70.380,NaN,NaN,NaN,NaN,NaN,30.0,Mountain Wing,Mobile,https://www.taalvistahotel.com/events/,2026-09-02
2,LOWER LEVEL FOYER,22.20 X 5.45,3.40,120.996,60.0,NaN,NaN,NaN,30.0,30.0,Mountain Wing,Mobile,https://www.taalvistahotel.com/events/,2026-09-02
3,WALING-WALING 1,7.60 X 7.10,3.25,53.960,24.0,16.0,13.0,14.0,15.0,15.0,Mountain Wing,Mobile,https://www.taalvistahotel.com/events/,2026-09-02
4,WALING-WALING 2,7.40 X 7.10,3.25,52.540,24.0,16.0,13.0,14.0,15.0,15.0,Mountain Wing,Mobile,https://www.taalvistahotel.com/events/,2026-09-02


In [21]:
event_file_names = [
    "taal_vista_events_lake_desktop_raw.csv",
    "taal_vista_events_lake_mobile_raw.csv",
    "taal_vista_events_mountain_desktop_raw.csv",
    "taal_vista_events_mountain_mobile_raw.csv"
]

for file_name in event_file_names:
    print()
    print("=" * 70)
    print("FILE:", file_name)

    file_path = (
        raw_data_folder
        / file_name
    )

    with open(
        file_path,
        "r",
        encoding="utf-8"
    ) as file:
        for line_number in range(4):
            print(
                repr(file.readline().strip())
            )


FILE: taal_vista_events_lake_desktop_raw.csv
'0,1,2,3,4,5,6,7,8,9,hotel_wing,display_version,source_url,date_collected'
'VENUE,DIMENSION (IN METERS),CEILING HEIGHT (IN METERS),FLOOR AREA (IN SQ. METERS),CONFERENCE CAPACITY,CONFERENCE CAPACITY,CONFERENCE CAPACITY,CONFERENCE CAPACITY,BANQUET,COCKTAIL,Lake Wing,Desktop,https://www.taalvistahotel.com/events/,2026-09-02'
'VENUE,DIMENSION (IN METERS),CEILING HEIGHT (IN METERS),FLOOR AREA (IN SQ. METERS),THEATER,CLASSROOM,U-SHAPE,BLOCK,BANQUET,COCKTAIL,Lake Wing,Desktop,https://www.taalvistahotel.com/events/,2026-09-02'
'GRAND BALLROOM,43.80 X 29.50,5.50,1292.10,1000,750,200,260,800,1400,Lake Wing,Desktop,https://www.taalvistahotel.com/events/,2026-09-02'

FILE: taal_vista_events_lake_mobile_raw.csv
'VENUE,DIMENSION (IN METERS),FLOOR AREA (IN SQ. METERS),CONFERENCE CAPACITY,THEATER,CLASSROOM,U-SHAPE,BLOCK,BANQUET,COCKTAIL,hotel_wing,display_version,source_url,date_collected'
'GRAND BALLROOM,43.80 X 29.50,5.5,1292.1,360,180.0,90.0,90.0,270,27

In [22]:
standard_event_columns = [
    "venue",
    "dimension_meters",
    "ceiling_height_meters",
    "floor_area_sqm",
    "theater_capacity",
    "classroom_capacity",
    "u_shape_capacity",
    "block_capacity",
    "banquet_capacity",
    "cocktail_capacity",
    "hotel_wing",
    "display_version",
    "source_url",
    "date_collected"
]

desktop_event_names = [
    "events_lake_desktop",
    "events_mountain_desktop"
]

for dataset_name in desktop_event_names:
    current_df = (
        clean_dataframes[
            dataset_name
        ]
        .iloc[2:]
        .copy()
        .reset_index(
            drop=True
        )
    )

    current_df.columns = (
        standard_event_columns
    )

    clean_dataframes[
        dataset_name
    ] = current_df

In [23]:
mobile_event_names = [
    "events_lake_mobile",
    "events_mountain_mobile"
]

for dataset_name in mobile_event_names:
    current_df = (
        clean_dataframes[
            dataset_name
        ].copy()
    )

    current_df.columns = (
        standard_event_columns
    )

    clean_dataframes[
        dataset_name
    ] = current_df

In [24]:
event_numeric_columns = [
    "ceiling_height_meters",
    "floor_area_sqm",
    "theater_capacity",
    "classroom_capacity",
    "u_shape_capacity",
    "block_capacity",
    "banquet_capacity",
    "cocktail_capacity"
]

event_conversion_records = []

for dataset_name in event_dataset_names:
    current_df = clean_dataframes[
        dataset_name
    ]

    for column_name in event_numeric_columns:
        original_missing = (
            current_df[column_name]
            .isna()
            .sum()
        )

        current_df[column_name] = (
            pd.to_numeric(
                current_df[column_name],
                errors="coerce"
            )
        )

        final_missing = (
            current_df[column_name]
            .isna()
            .sum()
        )

        event_conversion_records.append({
            "dataset_name": dataset_name,
            "column_name": column_name,
            "invalid_values_created": (
                final_missing
                - original_missing
            )
        })

event_conversion_df = pd.DataFrame(
    event_conversion_records
)

print(
    "Invalid numeric values created:",
    event_conversion_df[
        "invalid_values_created"
    ].sum()
)

Invalid numeric values created: 0


In [25]:
events_clean_df = pd.concat(
    [
        clean_dataframes[
            "events_lake_desktop"
        ],
        clean_dataframes[
            "events_lake_mobile"
        ],
        clean_dataframes[
            "events_mountain_desktop"
        ],
        clean_dataframes[
            "events_mountain_mobile"
        ]
    ],
    ignore_index=True
)

events_clean_df.insert(
    0,
    "event_record_id",
    range(
        1,
        len(events_clean_df) + 1
    )
)

print(
    "Combined event shape:",
    events_clean_df.shape
)

print(
    "\nRecords by wing and version:"
)

print(
    events_clean_df.groupby(
        [
            "hotel_wing",
            "display_version"
        ]
    ).size()
)

print(
    "\nDuplicate observations:"
)

print(
    events_clean_df.duplicated(
        subset=[
            "venue",
            "hotel_wing",
            "display_version"
        ]
    ).sum()
)

print(
    "\nInvalid floor areas:"
)

print(
    (
        events_clean_df[
            "floor_area_sqm"
        ] <= 0
    ).sum()
)

Combined event shape: (47, 15)

Records by wing and version:
hotel_wing     display_version
Lake Wing      Desktop            15
               Mobile             14
Mountain Wing  Desktop             9
               Mobile              9
dtype: int64

Duplicate observations:
0

Invalid floor areas:
0


In [26]:
event_desktop_df = (
    events_clean_df[
        events_clean_df[
            "display_version"
        ] == "Desktop"
    ]
    .copy()
)

event_mobile_df = (
    events_clean_df[
        events_clean_df[
            "display_version"
        ] == "Mobile"
    ]
    .copy()
)

event_comparison_df = (
    event_desktop_df.merge(
        event_mobile_df,
        on=[
            "venue",
            "hotel_wing"
        ],
        how="outer",
        suffixes=(
            "_desktop",
            "_mobile"
        ),
        indicator=True
    )
)

print(
    "Venue comparison records:",
    event_comparison_df.shape[0]
)

print(
    "\nVenue matching status:"
)

print(
    event_comparison_df[
        "_merge"
    ].value_counts()
)

Venue comparison records: 24

Venue matching status:
_merge
both          23
left_only      1
right_only     0
Name: count, dtype: int64


In [27]:
unmatched_event_venues_df = (
    event_comparison_df[
        event_comparison_df["_merge"]
        != "both"
    ][
        [
            "venue",
            "hotel_wing",
            "_merge"
        ]
    ]
)

display(unmatched_event_venues_df)

,venue,hotel_wing,_merge
14,MID GARDEN,Lake Wing,left_only


In [28]:
capacity_columns = [
    "theater_capacity",
    "classroom_capacity",
    "u_shape_capacity",
    "block_capacity",
    "banquet_capacity",
    "cocktail_capacity"
]

for column_name in capacity_columns:
    desktop_column = (
        column_name
        + "_desktop"
    )

    mobile_column = (
        column_name
        + "_mobile"
    )

    conflict_column = (
        column_name
        + "_conflict"
    )

    event_comparison_df[
        conflict_column
    ] = (
        event_comparison_df[
            desktop_column
        ].fillna(-1)
        != event_comparison_df[
            mobile_column
        ].fillna(-1)
    )

capacity_conflict_columns = [
    column_name + "_conflict"
    for column_name in capacity_columns
]

event_comparison_df[
    "any_capacity_conflict"
] = (
    event_comparison_df[
        capacity_conflict_columns
    ].any(
        axis=1
    )
)

matched_event_comparison_df = (
    event_comparison_df[
        event_comparison_df["_merge"]
        == "both"
    ]
)

print(
    "Matched venues:",
    matched_event_comparison_df.shape[0]
)

print(
    "Matched venues with capacity conflicts:",
    matched_event_comparison_df[
        "any_capacity_conflict"
    ].sum()
)

print(
    "\nConflicts by capacity type:"
)

print(
    matched_event_comparison_df[
        capacity_conflict_columns
    ].sum()
)

Matched venues: 23
Matched venues with capacity conflicts: 23

Conflicts by capacity type:
theater_capacity_conflict      21
classroom_capacity_conflict    17
u_shape_capacity_conflict      17
block_capacity_conflict        18
banquet_capacity_conflict      21
cocktail_capacity_conflict     22
dtype: int64


In [29]:
event_conflict_summary_df = (
    event_comparison_df[
        [
            "venue",
            "hotel_wing",
            "_merge",
            "any_capacity_conflict"
        ]
    ]
    .copy()
)

event_conflict_summary_df[
    "venue_version_status"
] = (
    event_conflict_summary_df[
        "_merge"
    ].map({
        "both": "Desktop and Mobile",
        "left_only": "Desktop only",
        "right_only": "Mobile only"
    })
)

event_conflict_summary_df[
    "capacity_conflict"
] = (
    event_conflict_summary_df[
        "any_capacity_conflict"
    ]
    .where(
        event_conflict_summary_df[
            "_merge"
        ] == "both",
        pd.NA
    )
    .astype("boolean")
)

event_conflict_summary_df = (
    event_conflict_summary_df[
        [
            "venue",
            "hotel_wing",
            "venue_version_status",
            "capacity_conflict"
        ]
    ]
)

events_clean_df = (
    events_clean_df.merge(
        event_conflict_summary_df,
        on=[
            "venue",
            "hotel_wing"
        ],
        how="left"
    )
)

print(
    "Updated event shape:",
    events_clean_df.shape
)

print(
    "\nVersion status:"
)

print(
    events_clean_df[
        "venue_version_status"
    ].value_counts()
)

print(
    "\nCapacity conflict status:"
)

print(
    events_clean_df[
        "capacity_conflict"
    ].value_counts(
        dropna=False
    )
)

Updated event shape: (47, 17)

Version status:
venue_version_status
Desktop and Mobile    46
Desktop only           1
Mobile only            0
Name: count, dtype: int64

Capacity conflict status:
capacity_conflict
True    46
<NA>     1
Name: count, dtype: Int64


### Event Capacity Cleaning Decision

The desktop and mobile versions of the official events page display different capacity values for every venue appearing in both versions. These differences affect theater, classroom, U shape, block, banquet, and cocktail arrangements.

The conflicting values were retained because there is insufficient evidence to determine which display version is authoritative. Each observation retains its display version, and a capacity conflict indicator was added.

Mid Garden appears only in the desktop version. Its capacity conflict status remains missing because no mobile observation is available for comparison.

Business users should verify current venue capacities directly with the hotel before preparing proposals, allocating space, or confirming an event booking.

In [30]:
property_comparison_columns = [
    "dimension_meters",
    "ceiling_height_meters",
    "floor_area_sqm"
]

for column_name in property_comparison_columns:
    desktop_column = (
        column_name
        + "_desktop"
    )

    mobile_column = (
        column_name
        + "_mobile"
    )

    matched_values = (
        event_comparison_df[
            event_comparison_df["_merge"]
            == "both"
        ]
    )

    conflict_count = (
        matched_values[
            desktop_column
        ].fillna(-1)
        != matched_values[
            mobile_column
        ].fillna(-1)
    ).sum()

    print(
        column_name,
        "conflicts:",
        conflict_count
    )

dimension_meters conflicts: 0
ceiling_height_meters conflicts: 0
floor_area_sqm conflicts: 0


In [31]:
event_aggregation_rules = {
    "dimension_meters": "first",
    "ceiling_height_meters": "max",
    "floor_area_sqm": "max",
    "theater_capacity": "max",
    "classroom_capacity": "max",
    "u_shape_capacity": "max",
    "block_capacity": "max",
    "banquet_capacity": "max",
    "cocktail_capacity": "max",
    "display_version": (
        lambda values: " | ".join(
            sorted(
                values.dropna().unique()
            )
        )
    ),
    "source_url": "first",
    "date_collected": "max",
    "venue_version_status": "first",
    "capacity_conflict": "first"
}

event_venues_clean_df = (
    events_clean_df.groupby(
        [
            "venue",
            "hotel_wing"
        ],
        as_index=False
    )
    .agg(
        event_aggregation_rules
    )
)

event_venues_clean_df = (
    event_venues_clean_df.rename(
        columns={
            "display_version":
            "source_versions"
        }
    )
)

event_venues_clean_df.insert(
    0,
    "venue_record_id",
    range(
        1,
        len(event_venues_clean_df) + 1
    )
)

event_venues_clean_df[
    "capacity_selection_method"
] = (
    "Maximum published capacity across official desktop and mobile tables"
)

print(
    "Consolidated shape:",
    event_venues_clean_df.shape
)

print(
    "Unique venues:",
    event_venues_clean_df[
        "venue"
    ].nunique()
)

print(
    "Duplicate venues:",
    event_venues_clean_df.duplicated(
        subset=[
            "venue",
            "hotel_wing"
        ]
    ).sum()
)

display(
    event_venues_clean_df.head()
)

Consolidated shape: (24, 18)
Unique venues: 24
Duplicate venues: 0


,venue_record_id,venue,hotel_wing,dimension_meters,ceiling_height_meters,floor_area_sqm,theater_capacity,classroom_capacity,u_shape_capacity,block_capacity,banquet_capacity,cocktail_capacity,source_versions,source_url,date_collected,venue_version_status,capacity_conflict,capacity_selection_method
0,1,BALLROOM 1,Lake Wing,29.50 X 14.60,5.50,430.7,350.0,220.0,80.0,100.0,220.0,450.0,Desktop | Mobile,https://www.taalvistahotel.com/events/,2026-09-02,Desktop and Mobile,True,Maximum published capacity across official des...
1,2,BALLROOM 2,Lake Wing,29.50 X 14.60,5.50,430.7,350.0,220.0,80.0,100.0,220.0,450.0,Desktop | Mobile,https://www.taalvistahotel.com/events/,2026-09-02,Desktop and Mobile,True,Maximum published capacity across official des...
2,3,BALLROOM 3,Lake Wing,29.50 X 14.60,5.50,430.7,350.0,220.0,80.0,100.0,220.0,450.0,Desktop | Mobile,https://www.taalvistahotel.com/events/,2026-09-02,Desktop and Mobile,True,Maximum published capacity across official des...
3,4,CAMIA,Lake Wing,8.50 X 6.80,3.40,57.8,50.0,32.0,22.0,22.0,30.0,60.0,Desktop | Mobile,https://www.taalvistahotel.com/events/,2026-09-02,Desktop and Mobile,True,Maximum published capacity across official des...
4,5,CATTLEYA,Lake Wing,7.70 X 6.35,3.25,48.9,40.0,30.0,20.0,20.0,20.0,50.0,Desktop | Mobile,https://www.taalvistahotel.com/events/,2026-09-02,Desktop and Mobile,True,Maximum published capacity across official des...


In [32]:
for capacity_column in capacity_columns:
    best_venue = (
        event_venues_clean_df.loc[
            event_venues_clean_df[
                capacity_column
            ].idxmax()
        ]
    )

    print()
    print(
        capacity_column.replace(
            "_",
            " "
        ).title()
    )

    print(
        "Venue:",
        best_venue["venue"]
    )

    print(
        "Wing:",
        best_venue["hotel_wing"]
    )

    print(
        "Maximum published capacity:",
        best_venue[
            capacity_column
        ]
    )


Theater Capacity
Venue: GRAND BALLROOM
Wing: Lake Wing
Maximum published capacity: 1000.0

Classroom Capacity
Venue: GRAND BALLROOM
Wing: Lake Wing
Maximum published capacity: 750.0

U Shape Capacity
Venue: GRAND BALLROOM
Wing: Lake Wing
Maximum published capacity: 200.0

Block Capacity
Venue: GRAND BALLROOM
Wing: Lake Wing
Maximum published capacity: 260.0

Banquet Capacity
Venue: GRAND BALLROOM
Wing: Lake Wing
Maximum published capacity: 800.0

Cocktail Capacity
Venue: GRAND BALLROOM
Wing: Lake Wing
Maximum published capacity: 1400.0


### Consolidated Event Venue Result

The desktop and mobile event tables were consolidated into 24 unique venues. Venue dimensions, ceiling heights, and floor areas were consistent across both versions.

For capacity fields, the highest value published across the desktop and mobile versions was retained. This provides a practical maximum published capacity for venue comparison while preserving the capacity conflict indicator.

The Grand Ballroom has the highest published capacity across all seating arrangements:

Theater: 1,000 guests  
Classroom: 750 guests  
U Shape: 200 guests  
Block: 260 guests  
Banquet: 800 guests  
Cocktail: 1,400 guests

These values support preliminary venue selection but should be confirmed directly with the hotel before final event planning.

In [33]:
clean_dataframes[
    "event_venues"
] = event_venues_clean_df

In [34]:
print(
    "ROOM INVENTORY NAMES"
)

print(
    clean_dataframes["rooms"][
        "room_name"
    ].sort_values().tolist()
)

print()
print(
    "ROOM RATE NAMES"
)

print(
    sorted(
        clean_dataframes["room_rates"][
            "room_name"
        ].unique()
    )
)

ROOM INVENTORY NAMES
['Batangas Suite', 'DELUXE ROOM', 'ONE-BEDROOM DELUXE SUITE', 'PREMIER QUEEN ROOM', 'PREMIER ROOM', 'TAAL SUITE', 'TAGAYTAY SUITE', 'Two-Bedroom Deluxe Suite', 'deluxe room', 'ridge room', 'superior room']

ROOM RATE NAMES
['Batangas Suite Two Bedroom', 'Deluxe King', 'Deluxe Mountain', 'Deluxe Queen', 'Deluxe Suite Mountain (2 Bedroom)', 'Deluxe Suite Mountain Two Bedroom', 'Deluxe Suite One Bedroom', 'Deluxe Suite Two Bedroom', 'Deluxe Twin', 'Premier Queen', 'Presidential Villa', 'Ridge', 'Superior King', 'Superior Twin', 'Taal Suite One Bedroom', 'Taal Suite Two Bedroom', 'Tagaytay Suite One Bedroom', 'Tagaytay Suite Two Bedroom']


In [35]:
display(
    clean_dataframes["rooms"][
        [
            "room_record_id",
            "room_name",
            "room_description"
        ]
    ]
)

,room_record_id,room_name,room_description
0,1,DELUXE ROOM,"Stay in our newly renovated Deluxe Rooms, thou..."
1,2,PREMIER QUEEN ROOM,Unwind in the newly renovated Premier Queen Ro...
2,3,Two-Bedroom Deluxe Suite,"Ideal for families or small groups, the newly ..."
3,4,Batangas Suite,Celebrate life’s milestones or simply unwind i...
4,5,superior room,Surrounded by the refreshing view of lush gree...
5,6,deluxe room,"Located on a higher floor, the rooms are taste..."
6,7,ridge room,Inspired by the rich culture and relaxing outd...
7,8,PREMIER ROOM,"For a more captivating and serene stay, our Pr..."
8,9,ONE-BEDROOM DELUXE SUITE,Features a spacious living room and the privac...
9,10,TAAL SUITE,Features a modern and fresh vibe with floor-to...


In [36]:
room_rate_name_audit_df = (
    clean_dataframes["room_rates"]
    .groupby(
        [
            "room_name",
            "search_id"
        ]
    )
    .size()
    .reset_index(
        name="records"
    )
    .sort_values(
        [
            "room_name",
            "search_id"
        ]
    )
)

display(room_rate_name_audit_df)

,room_name,search_id,records
0,Batangas Suite Two Bedroom,3,1
1,Batangas Suite Two Bedroom,4,1
2,Deluxe King,1,1
3,Deluxe King,2,3
4,Deluxe King,3,2
...,...,...,...
57,Tagaytay Suite One Bedroom,4,1
58,Tagaytay Suite Two Bedroom,1,1
59,Tagaytay Suite Two Bedroom,2,2
60,Tagaytay Suite Two Bedroom,3,1


In [37]:
suite_name_audit_df = (
    clean_dataframes["room_rates"][
        clean_dataframes[
            "room_rates"
        ]["room_name"]
        .str.contains(
            "Suite",
            case=False,
            na=False
        )
    ][
        [
            "search_id",
            "room_name",
            "room_size_sqm",
            "room_size_note"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        [
            "room_name",
            "search_id"
        ]
    )
)

display(suite_name_audit_df)

,search_id,room_name,room_size_sqm,room_size_note
64,3,Batangas Suite Two Bedroom,112.0,NaN
89,4,Batangas Suite Two Bedroom,112.0,NaN
42,2,Deluxe Suite Mountain (2 Bedroom),92.0,NaN
60,3,Deluxe Suite Mountain (2 Bedroom),92.0,NaN
85,4,Deluxe Suite Mountain (2 Bedroom),92.0,NaN
9,1,Deluxe Suite Mountain Two Bedroom,92.0,NaN
8,1,Deluxe Suite One Bedroom,62.0,NaN
40,2,Deluxe Suite One Bedroom,62.0,NaN
59,3,Deluxe Suite One Bedroom,62.0,NaN
84,4,Deluxe Suite One Bedroom,62.0,NaN


In [38]:
room_rates_clean_df = (
    clean_dataframes[
        "room_rates"
    ]
)

room_rates_clean_df[
    "original_room_name"
] = (
    room_rates_clean_df[
        "room_name"
    ]
)

room_name_mapping = {
    "Deluxe Suite Mountain Two Bedroom":
    "Deluxe Suite Mountain (2 Bedroom)"
}

room_rates_clean_df[
    "room_name"
] = (
    room_rates_clean_df[
        "room_name"
    ].replace(
        room_name_mapping
    )
)

print(
    "Original room names:",
    room_rates_clean_df[
        "original_room_name"
    ].nunique()
)

print(
    "Standardized room names:",
    room_rates_clean_df[
        "room_name"
    ].nunique()
)

print(
    "\nRecords using standardized names:"
)

display(
    room_rates_clean_df[
        room_rates_clean_df[
            "original_room_name"
        ]
        != room_rates_clean_df[
            "room_name"
        ]
    ][
        [
            "search_id",
            "original_room_name",
            "room_name",
            "room_size_sqm"
        ]
    ]
)

Original room names: 18
Standardized room names: 17

Records using standardized names:


,search_id,original_room_name,room_name,room_size_sqm
9,1,Deluxe Suite Mountain Two Bedroom,Deluxe Suite Mountain (2 Bedroom),92.0


In [39]:
display(
    clean_dataframes["rooms"][
        [
            "room_record_id",
            "room_name",
            "room_description"
        ]
    ]
)

,room_record_id,room_name,room_description
0,1,DELUXE ROOM,"Stay in our newly renovated Deluxe Rooms, thou..."
1,2,PREMIER QUEEN ROOM,Unwind in the newly renovated Premier Queen Ro...
2,3,Two-Bedroom Deluxe Suite,"Ideal for families or small groups, the newly ..."
3,4,Batangas Suite,Celebrate life’s milestones or simply unwind i...
4,5,superior room,Surrounded by the refreshing view of lush gree...
5,6,deluxe room,"Located on a higher floor, the rooms are taste..."
6,7,ridge room,Inspired by the rich culture and relaxing outd...
7,8,PREMIER ROOM,"For a more captivating and serene stay, our Pr..."
8,9,ONE-BEDROOM DELUXE SUITE,Features a spacious living room and the privac...
9,10,TAAL SUITE,Features a modern and fresh vibe with floor-to...


In [40]:
room_rates_clean_df[
    "rate_information"
] = (
    room_rates_clean_df[
        "rate_details_text"
    ].fillna(
        room_rates_clean_df[
            "rate_inclusions"
        ]
    )
)

print(
    "Missing rate information:",
    room_rates_clean_df[
        "rate_information"
    ].isna().sum()
)

Missing rate information: 0


In [41]:
clean_dataframes["dining"][
    "tagline"
] = (
    clean_dataframes["dining"][
        "tagline"
    ].fillna(
        "Not displayed"
    )
)

clean_dataframes["dining"][
    "schedule_note"
] = (
    clean_dataframes["dining"][
        "schedule_note"
    ].fillna(
        "Not displayed"
    )
)

clean_dataframes["hotel_contact"][
    "availability"
] = (
    clean_dataframes["hotel_contact"][
        "availability"
    ].fillna(
        "Not displayed"
    )
)

clean_dataframes["hotel_facts"][
    "fact_unit"
] = (
    clean_dataframes["hotel_facts"][
        "fact_unit"
    ].fillna(
        "Not applicable"
    )
)

clean_dataframes["official_reviews"][
    "reviewer_location"
] = (
    clean_dataframes["official_reviews"][
        "reviewer_location"
    ].fillna(
        "Not displayed"
    )
)

clean_dataframes["pasalubong_prices"][
    "package_detail"
] = (
    clean_dataframes[
        "pasalubong_prices"
    ]["package_detail"]
    .fillna(
        "Individual item"
    )
)

clean_dataframes["promotions"][
    "action_label"
] = (
    clean_dataframes["promotions"][
        "action_label"
    ].fillna(
        "No action displayed"
    )
)

In [42]:
room_rates_clean_df[
    "rate_information"
] = (
    room_rates_clean_df[
        "rate_details_text"
    ].fillna(
        room_rates_clean_df[
            "rate_inclusions"
        ]
    )
)

room_rates_clean_df[
    "availability_note"
] = (
    room_rates_clean_df[
        "availability_note"
    ].fillna(
        "No limited availability message displayed"
    )
)

room_rates_clean_df[
    "room_size_note"
] = (
    room_rates_clean_df[
        "room_size_note"
    ].fillna(
        "No additional room size note displayed"
    )
)

In [43]:
room_rates_clean_df = (
    room_rates_clean_df.drop(
        columns=[
            "collection_timestamp",
            "rate_inclusions",
            "rate_details_text"
        ]
    )
)

clean_dataframes[
    "room_rates"
] = room_rates_clean_df

In [44]:
clean_dataframes["meeting_rates"][
    "open_ended_participant_range"
] = (
    clean_dataframes[
        "meeting_rates"
    ]["maximum_participants"]
    .isna()
)

In [45]:
clean_dataframes["wedding_rates"][
    "fixed_ceremony_package"
] = (
    clean_dataframes[
        "wedding_rates"
    ]["package_category"]
    .eq("Ceremony")
)

In [46]:
clean_dataframes["dining"][
    "reservation_url_available"
] = (
    clean_dataframes[
        "dining"
    ]["reservation_url"]
    .notna()
)

clean_dataframes["promotions"][
    "details_url_available"
] = (
    clean_dataframes[
        "promotions"
    ]["details_url"]
    .notna()
)

clean_dataframes["promotions"][
    "action_url_available"
] = (
    clean_dataframes[
        "promotions"
    ]["action_url"]
    .notna()
)

In [47]:
remaining_missing_records = []

for dataset_name, current_df in (
    clean_dataframes.items()
):
    if dataset_name in event_dataset_names:
        continue

    for column_name in current_df.columns:
        missing_count = (
            current_df[column_name]
            .isna()
            .sum()
        )

        if missing_count > 0:
            remaining_missing_records.append({
                "dataset_name": dataset_name,
                "column_name": column_name,
                "missing_values": missing_count
            })

remaining_missing_df = pd.DataFrame(
    remaining_missing_records
)

display(remaining_missing_df)

,dataset_name,column_name,missing_values
0,dining,reservation_url,3
1,meeting_rates,maximum_participants,6
2,promotions,details_url,11
3,promotions,action_url,9
4,wedding_rates,included_persons,1
5,wedding_rates,excess_person_rate_php,1
6,wedding_rates,excess_person_rate_eur,1
7,wedding_rates,excess_person_rate_usd,1
8,event_venues,ceiling_height_meters,3
9,event_venues,theater_capacity,2


### Missing Value Treatment

Missing descriptive values were replaced only when the source meaning was clear. Examples include Not displayed, Not applicable, and Individual item.

Missing URLs remain empty because no destination was provided by the source. Missing numeric values also remain empty because replacing them with zero or an average would create inaccurate capacities, participant limits, or prices.

The meeting rate records with no maximum participant value represent open ended ranges of 61 guests and above. The wedding ceremony package has no included guest count or excess person rate because it uses a fixed package price.

Missing event capacity values indicate that the seating arrangement or measurement was not published. They are not zero capacity values.

All remaining missing values are therefore intentional and documented.

In [49]:
missing_value_treatment_records = [
    {
        "dataset_name": "dining",
        "column_name": "reservation_url",
        "classification": "Valid missing",
        "treatment": "Retained as missing",
        "reason": "No reservation URL displayed"
    },
    {
        "dataset_name": "meeting_rates",
        "column_name": "maximum_participants",
        "classification": "Structural missing",
        "treatment": "Retained as missing",
        "reason": "Open ended range of 61 and above"
    },
    {
        "dataset_name": "promotions",
        "column_name": "details_url",
        "classification": "Valid missing",
        "treatment": "Retained as missing",
        "reason": "No details URL displayed"
    },
    {
        "dataset_name": "promotions",
        "column_name": "action_url",
        "classification": "Valid missing",
        "treatment": "Retained as missing",
        "reason": "No action URL displayed"
    },
    {
        "dataset_name": "wedding_rates",
        "column_name": "included_persons",
        "classification": "Not applicable",
        "treatment": "Retained as missing",
        "reason": "Fixed ceremony package"
    },
    {
        "dataset_name": "wedding_rates",
        "column_name": "excess_person_rate",
        "classification": "Not applicable",
        "treatment": "Retained as missing",
        "reason": "Fixed ceremony package"
    },
    {
        "dataset_name": "event_venues",
        "column_name": "ceiling_height_meters",
        "classification": "Not published",
        "treatment": "Retained as missing",
        "reason": "No ceiling height displayed"
    },
    {
        "dataset_name": "event_venues",
        "column_name": "capacity_fields",
        "classification": "Not published",
        "treatment": "Retained as missing",
        "reason": "Arrangement capacity unavailable"
    },
    {
        "dataset_name": "event_venues",
        "column_name": "capacity_conflict",
        "classification": "Not comparable",
        "treatment": "Retained as missing",
        "reason": "Mid Garden appears only in desktop version"
    }
]

missing_value_treatment_df = pd.DataFrame(
    missing_value_treatment_records
)

display(missing_value_treatment_df)

,dataset_name,column_name,classification,treatment,reason
0,dining,reservation_url,Valid missing,Retained as missing,No reservation URL displayed
1,meeting_rates,maximum_participants,Structural missing,Retained as missing,Open ended range of 61 and above
2,promotions,details_url,Valid missing,Retained as missing,No details URL displayed
3,promotions,action_url,Valid missing,Retained as missing,No action URL displayed
4,wedding_rates,included_persons,Not applicable,Retained as missing,Fixed ceremony package
5,wedding_rates,excess_person_rate,Not applicable,Retained as missing,Fixed ceremony package
6,event_venues,ceiling_height_meters,Not published,Retained as missing,No ceiling height displayed
7,event_venues,capacity_fields,Not published,Retained as missing,Arrangement capacity unavailable
8,event_venues,capacity_conflict,Not comparable,Retained as missing,Mid Garden appears only in desktop version


In [50]:
rooms_clean_df = (
    clean_dataframes[
        "rooms"
    ].copy()
)

rooms_clean_df[
    "original_room_name"
] = (
    rooms_clean_df[
        "room_name"
    ]
)

inventory_room_name_mapping = {
    "DELUXE ROOM": "Deluxe Room",
    "PREMIER QUEEN ROOM": "Premier Queen Room",
    "Two-Bedroom Deluxe Suite":
    "Two Bedroom Deluxe Suite",
    "superior room": "Superior Room",
    "deluxe room": "Deluxe Room",
    "ridge room": "Ridge Room",
    "PREMIER ROOM": "Premier Room",
    "ONE-BEDROOM DELUXE SUITE":
    "One Bedroom Deluxe Suite",
    "TAAL SUITE": "Taal Suite",
    "TAGAYTAY SUITE": "Tagaytay Suite"
}

rooms_clean_df[
    "room_name"
] = (
    rooms_clean_df[
        "room_name"
    ].replace(
        inventory_room_name_mapping
    )
)

In [51]:
rooms_clean_df[
    "room_description_variant"
] = "Standard description"

rooms_clean_df.loc[
    rooms_clean_df[
        "room_description"
    ].str.contains(
        "newly renovated",
        case=False,
        na=False
    ),
    "room_description_variant"
] = "Newly renovated"

rooms_clean_df.loc[
    rooms_clean_df[
        "room_description"
    ].str.contains(
        "higher floor",
        case=False,
        na=False
    ),
    "room_description_variant"
] = "Higher floor"

In [52]:
print(
    "Rows:",
    rooms_clean_df.shape[0]
)

print(
    "Unique standardized names:",
    rooms_clean_df[
        "room_name"
    ].nunique()
)

print(
    "Exact duplicate room records:",
    rooms_clean_df.duplicated(
        subset=[
            "room_name",
            "room_description"
        ]
    ).sum()
)

display(
    rooms_clean_df[
        [
            "room_record_id",
            "original_room_name",
            "room_name",
            "room_description_variant"
        ]
    ]
)

Rows: 11
Unique standardized names: 10
Exact duplicate room records: 0


,room_record_id,original_room_name,room_name,room_description_variant
0,1,DELUXE ROOM,Deluxe Room,Newly renovated
1,2,PREMIER QUEEN ROOM,Premier Queen Room,Newly renovated
2,3,Two-Bedroom Deluxe Suite,Two Bedroom Deluxe Suite,Newly renovated
3,4,Batangas Suite,Batangas Suite,Newly renovated
4,5,superior room,Superior Room,Standard description
5,6,deluxe room,Deluxe Room,Higher floor
6,7,ridge room,Ridge Room,Standard description
7,8,PREMIER ROOM,Premier Room,Standard description
8,9,ONE-BEDROOM DELUXE SUITE,One Bedroom Deluxe Suite,Standard description
9,10,TAAL SUITE,Taal Suite,Standard description


In [56]:
categorical_audit_records = []

categorical_keywords = [
    "name",
    "category",
    "type",
    "style",
    "wing",
    "version",
    "currency",
    "sector",
    "duration",
    "basis",
    "source_context",
    "displayed_source",
    "day_type"
]

for dataset_name, current_df in (
    clean_dataframes.items()
):
    if dataset_name in event_dataset_names:
        continue

    for column_name in current_df.columns:
        if not any(
            keyword in column_name
            for keyword in categorical_keywords
        ):
            continue

        if not (
            current_df[column_name].dtype
            == "object"
            or str(
                current_df[column_name].dtype
            ) == "str"
        ):
            continue

        category_values = (
            current_df[column_name]
            .dropna()
            .astype(str)
            .drop_duplicates()
        )

        case_groups = {}

        for value in category_values:
            normalized_value = (
                value.casefold().strip()
            )

            if normalized_value not in case_groups:
                case_groups[
                    normalized_value
                ] = []

            case_groups[
                normalized_value
            ].append(value)

        for normalized_value, variants in (
            case_groups.items()
        ):
            if len(variants) > 1:
                categorical_audit_records.append({
                    "dataset_name": dataset_name,
                    "column_name": column_name,
                    "normalized_value": normalized_value,
                    "original_variants": variants
                })

categorical_case_audit_df = pd.DataFrame(
    categorical_audit_records
)

print(
    "Case inconsistency groups:",
    categorical_case_audit_df.shape[0]
)

display(categorical_case_audit_df)

Case inconsistency groups: 1


,dataset_name,column_name,normalized_value,original_variants
0,rooms,original_room_name,deluxe room,"[DELUXE ROOM, deluxe room]"


In [57]:
final_clean_dataset_names = [
    "christmas_packages",
    "dining",
    "facilities",
    "hotel_contact",
    "hotel_facts",
    "meeting_rates",
    "official_reviews",
    "pasalubong_prices",
    "promotions",
    "room_rates",
    "rooms",
    "spa_prices",
    "wedding_rates",
    "event_venues"
]

final_clean_dataframes = {}

for dataset_name in (
    final_clean_dataset_names
):
    final_clean_dataframes[
        dataset_name
    ] = (
        clean_dataframes[
            dataset_name
        ].copy()
    )

print(
    "Final cleaned datasets:",
    len(final_clean_dataframes)
)

print(
    "Total cleaned rows:",
    sum(
        current_df.shape[0]
        for current_df
        in final_clean_dataframes.values()
    )
)

Final cleaned datasets: 14
Total cleaned rows: 277


In [58]:
clean_validation_records = []

for dataset_name, current_df in (
    final_clean_dataframes.items()
):
    record_id_columns = [
        column_name
        for column_name in current_df.columns
        if column_name.endswith(
            "_record_id"
        )
    ]

    if record_id_columns:
        record_id_column = (
            record_id_columns[0]
        )

        duplicate_ids = (
            current_df[
                record_id_column
            ].duplicated().sum()
        )
    else:
        record_id_column = None
        duplicate_ids = None

    clean_validation_records.append({
        "dataset_name": dataset_name,
        "rows": current_df.shape[0],
        "columns": current_df.shape[1],
        "missing_cells": (
            current_df.isna().sum().sum()
        ),
        "duplicate_rows": (
            current_df.duplicated().sum()
        ),
        "record_id_column": record_id_column,
        "duplicate_record_ids": duplicate_ids
    })

clean_validation_df = pd.DataFrame(
    clean_validation_records
)

display(clean_validation_df)

print(
    "Datasets with duplicate rows:",
    (
        clean_validation_df[
            "duplicate_rows"
        ] > 0
    ).sum()
)

print(
    "Datasets with duplicate record IDs:",
    (
        clean_validation_df[
            "duplicate_record_ids"
        ].fillna(0) > 0
    ).sum()
)

,dataset_name,rows,columns,missing_cells,duplicate_rows,record_id_column,duplicate_record_ids
0,christmas_packages,4,25,0,0,christmas_package_record_id,0
1,dining,5,11,3,0,outlet_record_id,0
2,facilities,5,6,0,0,facility_record_id,0
3,hotel_contact,5,8,0,0,contact_record_id,0
4,hotel_facts,12,8,0,0,fact_record_id,0
5,meeting_rates,18,21,6,0,meeting_rate_record_id,0
6,official_reviews,6,10,0,0,review_record_id,0
7,pasalubong_prices,34,17,0,0,menu_record_id,0
8,promotions,25,11,20,0,promotion_record_id,0
9,room_rates,93,38,0,0,room_rate_record_id,0


Datasets with duplicate rows: 0
Datasets with duplicate record IDs: 0


In [59]:
display(
    final_clean_dataframes["dining"].loc[
        final_clean_dataframes[
            "dining"
        ]["reservation_url"].isna(),
        [
            "outlet_record_id",
            "outlet_name",
            "reservation_url"
        ]
    ]
)

,outlet_record_id,outlet_name,reservation_url
2,3,LOBBY LOUNGE,NaN
3,4,pasalubong,NaN
4,5,alta ridge bar,NaN


In [60]:
display(
    final_clean_dataframes[
        "meeting_rates"
    ].loc[
        final_clean_dataframes[
            "meeting_rates"
        ]["maximum_participants"].isna(),
        [
            "meeting_rate_record_id",
            "client_sector",
            "participant_range",
            "minimum_participants",
            "maximum_participants",
            "open_ended_participant_range"
        ]
    ]
)

,meeting_rate_record_id,client_sector,participant_range,minimum_participants,maximum_participants,open_ended_participant_range
6,7,Government,61 and above,61,NaN,True
7,8,Government,61 and above,61,NaN,True
8,9,Government,61 and above,61,NaN,True
15,16,Corporate,61 and above,61,NaN,True
16,17,Corporate,61 and above,61,NaN,True
17,18,Corporate,61 and above,61,NaN,True


In [61]:
promotion_missing_urls_df = (
    final_clean_dataframes[
        "promotions"
    ].loc[
        final_clean_dataframes[
            "promotions"
        ][
            [
                "details_url",
                "action_url"
            ]
        ].isna().any(
            axis=1
        ),
        [
            "promotion_record_id",
            "promotion_category",
            "promotion_name",
            "details_url",
            "action_label",
            "action_url",
            "details_url_available",
            "action_url_available"
        ]
    ]
)

display(promotion_missing_urls_df)

,promotion_record_id,promotion_category,promotion_name,details_url,action_label,action_url,details_url_available,action_url_available
6,7,Dining,PLATE FOR THE PLANET,https://www.taalvistahotel.com/wp-content/uplo...,No action displayed,NaN,True,False
7,8,Dining,Perfect Pairings: Bites & Beer,https://www.taalvistahotel.com/wp-content/uplo...,No action displayed,NaN,True,False
8,9,Dining,DINE UNDER THE STARS,NaN,BOOK YOUR TABLE,https://bit.ly/TVHDiningReservations,False,True
9,10,Dining,GLAM PICNIC AT THE LAWN,NaN,RESERVE NOW,https://forms.office.com/r/LTwLn6BxuU,False,True
10,11,Dining,IN-ROOM DINING SPECIALS,NaN,No action displayed,NaN,False,False
11,12,Dining,CUISINE MEETS CULTURE,NaN,RESERVE NOW,https://forms.office.com/r/LTwLn6BxuU,False,True
12,13,Dining,GELATO,NaN,No action displayed,NaN,False,False
13,14,Dining,FILIPINO CLASSICS,https://www.taalvistahotel.com/wp-content/uplo...,No action displayed,NaN,True,False
14,15,Dining,GINGER GLAZED RIBS,NaN,No action displayed,NaN,False,False
15,16,Dining,A WALK THROUGH TIME,NaN,No action displayed,NaN,False,False


In [62]:
wedding_missing_df = (
    final_clean_dataframes[
        "wedding_rates"
    ].loc[
        final_clean_dataframes[
            "wedding_rates"
        ].isna().any(
            axis=1
        ),
        [
            "wedding_rate_record_id",
            "package_category",
            "package_name",
            "menu_style",
            "included_persons",
            "package_price_php",
            "excess_person_rate_php",
            "fixed_ceremony_package"
        ]
    ]
)

display(wedding_missing_df)

,wedding_rate_record_id,package_category,package_name,menu_style,included_persons,package_price_php,excess_person_rate_php,fixed_ceremony_package
8,9,Ceremony,Wedding Rites Package,Not applicable,NaN,100000,NaN,True


In [63]:
event_missing_columns = [
    "ceiling_height_meters",
    "theater_capacity",
    "classroom_capacity",
    "u_shape_capacity",
    "block_capacity",
    "banquet_capacity",
    "cocktail_capacity",
    "capacity_conflict"
]

event_missing_df = (
    final_clean_dataframes[
        "event_venues"
    ].loc[
        final_clean_dataframes[
            "event_venues"
        ][
            event_missing_columns
        ].isna().any(
            axis=1
        ),
        [
            "venue_record_id",
            "venue",
            "hotel_wing",
            "venue_version_status"
        ]
        + event_missing_columns
    ]
)

display(event_missing_df)

,venue_record_id,venue,hotel_wing,venue_version_status,ceiling_height_meters,theater_capacity,classroom_capacity,u_shape_capacity,block_capacity,banquet_capacity,cocktail_capacity,capacity_conflict
7,8,COVERED TERRACE,Lake Wing,Desktop and Mobile,3.90,300.0,NaN,NaN,NaN,220.0,450.0,True
10,11,GUMAMELA,Mountain Wing,Desktop and Mobile,3.20,NaN,NaN,NaN,12.0,NaN,NaN,True
13,14,LOWER LEVEL FOYER,Mountain Wing,Desktop and Mobile,3.40,100.0,NaN,NaN,NaN,60.0,120.0,True
14,15,MID GARDEN,Lake Wing,Desktop only,NaN,120.0,NaN,NaN,30.0,60.0,120.0,<NA>
15,16,PREMIER GARDEN,Lake Wing,Desktop and Mobile,NaN,300.0,NaN,NaN,NaN,200.0,430.0,True
19,20,SAMPAGUITA FOYER,Mountain Wing,Desktop and Mobile,3.88,NaN,NaN,NaN,NaN,NaN,70.0,True
21,22,VIEWDECK,Lake Wing,Desktop and Mobile,NaN,600.0,NaN,NaN,NaN,500.0,860.0,True


In [64]:
print(
    "Remaining missing cells:",
    sum(
        current_df.isna().sum().sum()
        for current_df
        in final_clean_dataframes.values()
    )
)

Remaining missing cells: 61


In [65]:
taal_vista_homepage_url = (
    "https://www.taalvistahotel.com/"
)

promotions_clean_df = (
    final_clean_dataframes[
        "promotions"
    ]
)

promotions_clean_df[
    "details_url_fallback_used"
] = (
    promotions_clean_df[
        "details_url"
    ].isna()
)

promotions_clean_df[
    "action_url_fallback_used"
] = (
    promotions_clean_df[
        "action_url"
    ].isna()
)

promotions_clean_df[
    "details_url"
] = (
    promotions_clean_df[
        "details_url"
    ].fillna(
        taal_vista_homepage_url
    )
)

promotions_clean_df[
    "action_url"
] = (
    promotions_clean_df[
        "action_url"
    ].fillna(
        taal_vista_homepage_url
    )
)

In [66]:
print(
    "Missing details URLs:",
    promotions_clean_df[
        "details_url"
    ].isna().sum()
)

print(
    "Missing action URLs:",
    promotions_clean_df[
        "action_url"
    ].isna().sum()
)

print(
    "Details URL fallbacks:",
    promotions_clean_df[
        "details_url_fallback_used"
    ].sum()
)

print(
    "Action URL fallbacks:",
    promotions_clean_df[
        "action_url_fallback_used"
    ].sum()
)

Missing details URLs: 0
Missing action URLs: 0
Details URL fallbacks: 11
Action URL fallbacks: 9


In [67]:
print(
    "FINAL CLEANING VALIDATION"
)

print()

print(
    "Cleaned datasets:",
    len(final_clean_dataframes)
)

print(
    "Total cleaned rows:",
    sum(
        current_df.shape[0]
        for current_df
        in final_clean_dataframes.values()
    )
)

print(
    "Total duplicate rows:",
    sum(
        current_df.duplicated().sum()
        for current_df
        in final_clean_dataframes.values()
    )
)

print(
    "Remaining missing cells:",
    sum(
        current_df.isna().sum().sum()
        for current_df
        in final_clean_dataframes.values()
    )
)

FINAL CLEANING VALIDATION

Cleaned datasets: 14
Total cleaned rows: 277
Total duplicate rows: 0
Remaining missing cells: 41


In [68]:
negative_value_records = []

for dataset_name, current_df in (
    final_clean_dataframes.items()
):
    numeric_columns = (
        current_df.select_dtypes(
            include="number"
        ).columns
    )

    for column_name in numeric_columns:
        if column_name.endswith(
            "_record_id"
        ):
            continue

        negative_count = (
            current_df[column_name] < 0
        ).sum()

        if negative_count > 0:
            negative_value_records.append({
                "dataset_name": dataset_name,
                "column_name": column_name,
                "negative_values": negative_count
            })

negative_value_df = pd.DataFrame(
    negative_value_records
)

print(
    "Columns with negative values:",
    negative_value_df.shape[0]
)

display(negative_value_df)

Columns with negative values: 0


""


In [69]:
room_rate_validation_df = (
    final_clean_dataframes[
        "room_rates"
    ]
)

calculated_nights = (
    room_rate_validation_df[
        "check_out_date"
    ]
    - room_rate_validation_df[
        "check_in_date"
    ]
).dt.days

print(
    "Invalid date sequences:",
    (
        room_rate_validation_df[
            "check_out_date"
        ]
        <= room_rate_validation_df[
            "check_in_date"
        ]
    ).sum()
)

print(
    "Incorrect night counts:",
    (
        calculated_nights
        != room_rate_validation_df[
            "number_of_nights"
        ]
    ).sum()
)

Invalid date sequences: 0
Incorrect night counts: 0


In [70]:
expected_room_rate_usd = (
    room_rate_validation_df[
        "rate_plan_price_php"
    ]
    * php_to_usd
).round(2)

expected_room_rate_eur = (
    room_rate_validation_df[
        "rate_plan_price_php"
    ]
    * php_to_eur
).round(2)

print(
    "Incorrect USD conversions:",
    (
        expected_room_rate_usd
        != room_rate_validation_df[
            "rate_plan_price_usd"
        ]
    ).sum()
)

print(
    "Incorrect EUR conversions:",
    (
        expected_room_rate_eur
        != room_rate_validation_df[
            "rate_plan_price_eur"
        ]
    ).sum()
)

Incorrect USD conversions: 0
Incorrect EUR conversions: 0
